In [45]:
import pandas as pd
import numpy as np
import re
import joblib
import nltk
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [46]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [47]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tabulator')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vegeshsai_boppana\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vegeshsai_boppana\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vegeshsai_boppana\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Error loading punkt_tabulator: Package 'punkt_tabulator'
[nltk_data]     not found in index
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\vegeshsai_boppana\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [48]:
df = pd.read_csv('../data/raw/train.csv')

# EDA

In [49]:
print(df.shape)
print(df.columns.tolist())
df.head()

(40000, 2)
['review', 'sentiment']


,review,sentiment
0,I caught this little gem totally by accident b...,positive
1,I can't believe that I let myself into this mo...,negative
2,*spoiler alert!* it just gets to me the nerve ...,negative
3,If there's one thing I've learnt from watching...,negative
4,"I remember when this was in theaters, reviews ...",negative


In [50]:
# lets check for the null values
print("=== Null Values ===")
print(df.isnull().sum())

=== Null Values ===
review       0
sentiment    0
dtype: int64


In [51]:
# lets cehck for the class distribution
print("\n=== Class Distribution ===")
print(df['sentiment'].value_counts())


=== Class Distribution ===
sentiment
positive    20000
negative    20000
Name: count, dtype: int64


In [52]:
df['review_length'] = df['review'].apply(lambda x: len(x.split()))

print("\n=== Review Length Stats ===")
print(df['review_length'].describe())


=== Review Length Stats ===
count    40000.000000
mean       231.362750
std        171.083908
min          4.000000
25%        126.000000
50%        173.000000
75%        282.000000
max       2470.000000
Name: review_length, dtype: float64


## EDA Summary

### Dataset Overview
- Total reviews: 40,000
- Features: review (text), sentiment (target)
- No null values found

### Class Distribution
- Positive: 20,000 (50%)
- Negative: 20,000 (50%)
- Perfectly balanced — no class imbalance handling needed

### Review Length (Word Count)
- Average: 231 words
- Shortest: 4 words
- Longest: 2,470 words
- 50% of reviews are under 173 words
- Wide range suggests diverse writing styles

### Key Conclusions
- Clean dataset, no missing data
- Balanced classes simplify modeling
- Text length varies significantly — TF-IDF handles this naturally

In [53]:
# 1. Class Distribution
fig1 = px.bar(
    x=['Positive', 'Negative'],
    y=[20000, 20000],
    color=['Positive', 'Negative'],
    title='Class Distribution',
    labels={'x': 'Sentiment', 'y': 'Count'},
    color_discrete_map={'Positive': '#2ecc71', 'Negative': '#e74c3c'}
)
fig1.show()

# 2. Review Length Distribution
fig2 = px.histogram(
    df,
    x='review_length',
    color='sentiment',
    nbins=50,
    title='Review Length Distribution by Sentiment',
    labels={'review_length': 'Word Count', 'count': 'Number of Reviews'},
    barmode='overlay',
    opacity=0.7
)
fig2.show()

# 3. Box plot - review length by sentiment
fig3 = px.box(
    df,
    x='sentiment',
    y='review_length',
    color='sentiment',
    title='Review Length by Sentiment',
    labels={'review_length': 'Word Count', 'sentiment': 'Sentiment'},
    color_discrete_map={'positive': '#2ecc71', 'negative': '#e74c3c'}
)
fig3.show()

### Visualization Conclusions
- Class distribution is perfectly balanced (50/50)
- Review length distribution is identical for both sentiments
- Review length is NOT a useful feature for sentiment prediction
- Most reviews are 50-300 words; distribution is right-skewed
- Outliers exist (up to 2470 words) but are rare

### Lets do the Top Words Analysis becasue the words in positive and negative reviews are actually different this is the core assumption our entire model is built on.




In [54]:
from collections import Counter

stop_words = set(stopwords.words('english'))

def get_top_words(reviews, n=20):
    words = []
    for review in reviews:
        # lowercase and remove special chars
        review = re.sub(r'<.*?>', '', review)  # remove HTML
        review = re.sub(r'[^a-zA-Z\s]', '', review.lower())
        tokens = review.split()
        # remove stopwords
        tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
        words.extend(tokens)
    return Counter(words).most_common(n)

In [55]:
positive_reviews = df[df['sentiment'] == 'positive']['review']
negative_reviews = df[df['sentiment'] == 'negative']['review']

top_pos = get_top_words(positive_reviews)
top_neg = get_top_words(negative_reviews)


In [56]:
print(top_pos)
print(top_neg)

[('film', 31785), ('movie', 28772), ('one', 20629), ('like', 13476), ('good', 11484), ('great', 10053), ('story', 9857), ('see', 9477), ('time', 9420), ('well', 8744), ('really', 8549), ('also', 8515), ('would', 8260), ('even', 7449), ('much', 7223), ('first', 7049), ('films', 6771), ('people', 6729), ('best', 6673), ('love', 6665)]
[('movie', 38128), ('film', 28128), ('one', 19760), ('like', 17447), ('even', 11916), ('bad', 11369), ('good', 11344), ('would', 10978), ('really', 9834), ('time', 9221), ('see', 8509), ('dont', 8029), ('get', 7935), ('much', 7845), ('story', 7778), ('could', 7243), ('people', 7208), ('make', 7154), ('movies', 6718), ('made', 6717)]


### Lets Viusalize these top words

In [57]:
fig_pos = px.bar(
    x=[w[0] for w in top_pos],
    y=[w[1] for w in top_pos],
    title='Top 20 Words in Positive Reviews',
    labels={'x': 'Word', 'y': 'Count'}
)
fig_pos.show()

fig_neg = px.bar(
    x=[w[0] for w in top_neg],
    y=[w[1] for w in top_neg],
    title='Top 20 Words in Negative Reviews',
    labels={'x': 'Word', 'y': 'Count'}
)
fig_neg.show()

## Top Words Analysis — Conclusions

### Unigram Observations
- Generic words like "film", "movie", "one", "like" dominate both classes
- These words carry no sentiment signal and overlap heavily
- Need to add "film", "movie", "movies", "films", "one" as custom stopwords

### Why Bigrams Instead of Trigrams?
- Unigrams are too generic — both classes look similar
- Trigrams are too sparse — appear very few times across 20k reviews
- Bigrams strike the right balance:
    - Capture sentiment phrases like "waste time", "highly recommend", "bad acting"
    - Appear frequently enough to be meaningful
    - Give clearer separation between positive and negative vocabulary

### Next Step
- Re-plot top words with custom stopwords added
- Plot top bigrams for positive and negative reviews separately

In [58]:
custom_stopwords = stop_words.union({
    'film', 'movie', 'movies', 'films', 'one', 'br'
})

def get_top_ngrams(reviews, n=20, ngram=2):
    ngrams_list = []
    for review in reviews:
        # Clean
        review = re.sub(r'<.*?>', '', review)
        review = re.sub(r'[^a-zA-Z\s]', '', review.lower())
        tokens = review.split()
        # Remove stopwords
        tokens = [t for t in tokens if t not in custom_stopwords and len(t) > 2]
        # Generate ngrams
        ngrams = [' '.join(tokens[i:i+ngram]) for i in range(len(tokens)-ngram+1)]
        ngrams_list.extend(ngrams)
    return Counter(ngrams_list).most_common(n)

In [59]:
top_pos_bigrams = get_top_ngrams(positive_reviews, ngram=2)
top_neg_bigrams = get_top_ngrams(negative_reviews, ngram=2)

print(top_pos_bigrams)
print(top_neg_bigrams)

[('ive seen', 868), ('even though', 793), ('ever seen', 743), ('first time', 713), ('dont know', 683), ('new york', 641), ('special effects', 597), ('years ago', 573), ('dont think', 465), ('well done', 460), ('real life', 454), ('must see', 448), ('pretty good', 438), ('really good', 433), ('years later', 421), ('love story', 403), ('highly recommend', 400), ('high school', 399), ('long time', 394), ('year old', 380)]
[('ever seen', 1285), ('special effects', 1111), ('waste time', 1086), ('looks like', 990), ('dont know', 952), ('ive seen', 896), ('much better', 781), ('worst ever', 756), ('look like', 754), ('even though', 679), ('low budget', 676), ('ive ever', 663), ('pretty much', 554), ('really bad', 548), ('dont think', 546), ('bad acting', 525), ('main character', 517), ('want see', 515), ('year old', 479), ('high school', 442)]


In [60]:
# Plot positive bigrams
fig_pos_bi = px.bar(
    x=[w[0] for w in top_pos_bigrams],
    y=[w[1] for w in top_pos_bigrams],
    title='Top 20 Bigrams in Positive Reviews',
    labels={'x': 'Bigram', 'y': 'Count'}
)
fig_pos_bi.show()

# Plot negative bigrams
fig_neg_bi = px.bar(
    x=[w[0] for w in top_neg_bigrams],
    y=[w[1] for w in top_neg_bigrams],
    title='Top 20 Bigrams in Negative Reviews',
    labels={'x': 'Bigram', 'y': 'Count'}
)
fig_neg_bi.show()

### Bigram Conclusions
- Bigrams show clear vocabulary separation between classes
- Positive signals: "well done", "must see", "highly recommend", "really good", "love story"
- Negative signals: "waste time", "worst ever", "bad acting", "really bad", "low budget"
- Some bigrams overlap: "ever seen", "dont know", "even though" — neutral context
- Confirms that TF-IDF with bigrams (ngram_range=(1,2)) will capture these phrases effectively

# Text PreProcessing

## Text Preprocessing Pipeline

### Steps

#### Step 1 - Remove HTML Tags
- Raw IMDb reviews often contain HTML tags like `<br />`, `<b>`, and `</b>`.

#### Step 2 - Lowercase
- "Great" and "great" are the same word
- Without this, vectorizer treats them as different tokens
- Reduces vocabulary size significantly

#### Step 3 - Remove Special Characters and Numbers
- Punctuation like "!!!", "..." adds noise


#### Step 4 - Tokenization
- Splits text into individual word tokens
- Required step — all NLP models work on tokens, not raw strings

#### Step 5 - Stopword Removal
- Words like "the", "is", "a", "was" appear equally in both classes
- Zero discriminative value for sentiment
- Custom stopwords added: "film", "movie", "movies", "films", "one", "br"
- These are generic movie-review words confirmed by our EDA

#### Step 6 - Stemming vs Lemmatization

##### What is Stemming?
- Crudely chops word suffixes using rules
- "running" → "run", "terrible" → "terribl", "movies" → "movi"
- Output is NOT always a real word
- Very fast

##### What is Lemmatization?
- Converts word to its dictionary base form
- "running" → "run", "terrible" → "terrible", "better" → "good"
- Output is ALWAYS a real word
- Slower — uses a dictionary lookup

##### When to Choose Stemming?
- Large datasets where speed matters
- When interpretability is not important
- Quick baseline experiments

##### When to Choose Lemmatization?
- When output needs to be human readable
- When accuracy is slightly more important than speed
- When words need to retain their meaning

##### Which Do We Use?
- We will run BOTH and compare accuracy
- Expected outcome: lemmatization wins slightly
- We will use the better performing one for final model

In [61]:
ps = PorterStemmer()
lemmatizer = WordNetLemmatizer()

ps = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text, method='lemmatize'):
    # Step 1 - Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Step 2 - Lowercase
    text = text.lower()
    
    # Step 3 - Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Step 4 - Tokenize
    tokens = word_tokenize(text)
    
    # Step 5 - Remove stopwords and short words
    tokens = [t for t in tokens if t not in custom_stopwords and len(t) > 2]
    
    # Step 6 - Stem or Lemmatize
    if method == 'stem':
        tokens = [ps.stem(t) for t in tokens]
    elif method == 'lemmatize':
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return ' '.join(tokens)

In [62]:
# lets test it in one review row first
sample = df['review'][0]
print("=== Original ===")
print(sample[:300])
print("\n=== After Preprocessing (Lemmatize) ===")
print(preprocess_text(sample, method='lemmatize')[:300])
print("\n=== After Preprocessing (Stem) ===")
print(preprocess_text(sample, method='stem')[:300])

=== Original ===
I caught this little gem totally by accident back in 1980 or '81. I was at a revival theatre to see two old silly sci-fi movies. The theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). Most were somewhat amusing but THIS came on and, with

=== After Preprocessing (Lemmatize) ===
caught little gem totally accident back revival theatre see two old silly scifi theatre packed full warning showed bunch scifi short spoof get mood somewhat amusing came within second audience hysteric biggest laugh came showed princess laia huge cinnamon bun instead hair head look camera give grim 

=== After Preprocessing (Stem) ===
caught littl gem total accid back reviv theatr see two old silli scifi theatr pack full warn show bunch scifi short spoof get mood somewhat amus came within second audienc hyster biggest laugh came show princess laia huge cinnamon bun instead hair head look camera give grim smile nod made even funni


### Conclusion
- Lemmatization preserves real words — better for TF-IDF vocabulary
- Stemming aggressively chops words — creates non-existent tokens
- We will apply BOTH to full dataset and compare model accuracy
- Expected winner: Lemmatization

In [63]:
# Apply Preprocessing to Full Dataset

print("Applying Lemmatization")
df['review_lemmatized'] = df['review'].apply(lambda x: preprocess_text(x, method='lemmatize'))
print("Lemmatization Done")

print("\nApplying Stemming")
df['review_stemmed'] = df['review'].apply(lambda x: preprocess_text(x, method='stem'))
print("Stemming Done")


Applying Lemmatization
Lemmatization Done

Applying Stemming
Stemming Done


In [64]:
print("\nSample comparison:")
print("\nOriginal:", df['review'][0][:100])
print("\nLemmatized:", df['review_lemmatized'][0][:100])
print("\nStemmed:", df['review_stemmed'][0][:100])


Sample comparison:

Original: I caught this little gem totally by accident back in 1980 or '81. I was at a revival theatre to see 

Lemmatized: caught little gem totally accident back revival theatre see two old silly scifi theatre packed full 

Stemmed: caught littl gem total accid back reviv theatr see two old silli scifi theatr pack full warn show bu


### Preprocessing Applied to Full Dataset
- Lemmatization and Stemming both applied to all 40,000 reviews
- Two new columns created: review_lemmatized, review_stemmed
- Lemmatized output contains real readable words 
- Stemmed output contains chopped non-real words 
- Both will be used for vectorization comparison in modeling

In [65]:
# Train/Test Split for both Lemm and Stemm techniques


# Encode target variable
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})


X_lemma = df['review_lemmatized']
X_stem = df['review_stemmed']
y = df['label']

X_lemma_train, X_lemma_test, y_train, y_test = train_test_split(X_lemma, y, test_size=0.2, random_state=42, stratify=y)

X_stem_train, X_stem_test, _, _ = train_test_split(X_stem, y, test_size=0.2, random_state=42, stratify=y)

print("=== Split Summary ===")
print(f"Total samples: {len(df)}")

print(f"\nTrain class distribution:")
print(y_train.value_counts())
print(f"\nTest class distribution:")
print(y_test.value_counts())

=== Split Summary ===
Total samples: 40000

Train class distribution:
label
0    16000
1    16000
Name: count, dtype: int64

Test class distribution:
label
1    4000
0    4000
Name: count, dtype: int64


## Vectorization

### What is Vectorization?
Converting raw text into numerical representation that ML models can understand.
Models cannot work with raw strings — they need numbers.

### Technique 1 — Bag of Words (CountVectorizer)
- Creates a vocabulary of all unique words across all reviews
- Represents each review as a vector of word counts
- Example:
    - Vocabulary: ["good", "bad", "movie", "terrible"]
    - "good movie good" → [2, 0, 1, 0]
- Limitation: treats all words equally, ignores word importance
- Does NOT capture word order or context

### Technique 2 — TF-IDF (Unigrams)
- TF (Term Frequency): how often a word appears in a review
- IDF (Inverse Document Frequency): penalizes words that appear in ALL reviews
- Words like "movie", "film" appear everywhere → low IDF score
- Words like "masterpiece", "unwatchable" are rare → high IDF score
- Better than BoW because it rewards discriminative words
- Example:
    - "good" appears in 30,000 reviews → low weight
    - "masterpiece" appears in 500 reviews → high weight

### Technique 3 — TF-IDF with Bigrams (Unigrams + Bigrams)
- Same as TF-IDF but also captures two-word phrases
- ngram_range=(1,2) → includes both single words and pairs
- Captures negation and sentiment phrases:
    - "not good", "waste time", "highly recommend", "bad acting"
- Our EDA confirmed these bigrams are strong sentiment signals

### Key Parameters We Use
- max_features=50000 → limit vocabulary to top 50,000 words
- min_df=2 → ignore words appearing in less than 2 reviews (noise)
- sublinear_tf=True → apply log normalization to term frequency (TF-IDF only)

### Why Stemmed vs Lemmatized?
- We will run TF-IDF on BOTH stemmed and lemmatized text
- Compare accuracy to determine which preprocessing works better
- Expected winner: Lemmatized (real words = better vocabulary)

In [66]:
# 1. Bag of Words
bow_vectorizer = CountVectorizer(max_features=50000, min_df=2)
X_bow_train = bow_vectorizer.fit_transform(X_lemma_train)
X_bow_test = bow_vectorizer.transform(X_lemma_test)

# 2. TF-IDF Unigrams
tfidf_vectorizer = TfidfVectorizer(max_features=50000, min_df=2, sublinear_tf=True)
X_tfidf_train = tfidf_vectorizer.fit_transform(X_lemma_train)
X_tfidf_test = tfidf_vectorizer.transform(X_lemma_test)

# 3. TF-IDF Bigrams
tfidf_bigram_vectorizer = TfidfVectorizer(max_features=50000, min_df=2, sublinear_tf=True, ngram_range=(1,2))
X_tfidf_bi_train = tfidf_bigram_vectorizer.fit_transform(X_lemma_train)
X_tfidf_bi_test = tfidf_bigram_vectorizer.transform(X_lemma_test)

# 4. TF-IDF on Stemmed text (for comparison)
tfidf_stem_vectorizer = TfidfVectorizer(max_features=50000, min_df=2, sublinear_tf=True)
X_tfidf_stem_train = tfidf_stem_vectorizer.fit_transform(X_stem_train)
X_tfidf_stem_test = tfidf_stem_vectorizer.transform(X_stem_test)

print("=== Vectorization Summary ===")
print(f"BoW matrix shape:           {X_bow_train.shape}")
print(f"TF-IDF matrix shape:        {X_tfidf_train.shape}")
print(f"TF-IDF Bigram matrix shape: {X_tfidf_bi_train.shape}")
print(f"TF-IDF Stem matrix shape:   {X_tfidf_stem_train.shape}")

=== Vectorization Summary ===
BoW matrix shape:           (32000, 50000)
TF-IDF matrix shape:        (32000, 50000)
TF-IDF Bigram matrix shape: (32000, 50000)
TF-IDF Stem matrix shape:   (32000, 42767)


### Vectorization Results
- All matrices: 32,000 reviews × features
- BoW vocabulary: 50,000 (hit cap)
- TF-IDF vocabulary: 50,000 (hit cap)
- TF-IDF Bigram vocabulary: 50,000 (hit cap — includes word pairs)
- TF-IDF Stem vocabulary: 42,767 (stemming reduces vocab naturally)
- Stemming merges word variants → smaller vocabulary than lemmatization

# Modelling

In [67]:
results = [] 

def evaluate(model_name, vectorizer_name, y_test, y_pred):
    acc = accuracy_score(y_test, y_pred)
    print(f"{vectorizer_name} + {model_name}: Accuracy = {acc:.4f}")
    result = {
        'model_name': model_name,
        'vectorizer': vectorizer_name,
        'accuracy': acc,
        'y_pred': y_pred
    }
    results.append(result)
    return result

### Bag of Words Vectorization Technique

In [68]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_bow_train, y_train)
y_pred = lr.predict(X_bow_test)


evaluate('Logistic Regression', 'BoW', y_test, y_pred)

BoW + Logistic Regression: Accuracy = 0.8788


{'model_name': 'Logistic Regression',
 'vectorizer': 'BoW',
 'accuracy': 0.87875,
 'y_pred': array([1, 0, 1, ..., 1, 0, 0], dtype=int64)}

In [69]:
nb = MultinomialNB()
nb.fit(X_bow_train, y_train)
y_pred = nb.predict(X_bow_test)
evaluate('Naive Bayes', 'BoW', y_test, y_pred)

BoW + Naive Bayes: Accuracy = 0.8568


{'model_name': 'Naive Bayes',
 'vectorizer': 'BoW',
 'accuracy': 0.85675,
 'y_pred': array([1, 0, 1, ..., 1, 1, 0], dtype=int64)}

In [70]:
svc = LinearSVC(random_state=42, max_iter=10000, dual='auto')
svc.fit(X_bow_train, y_train)
y_pred = svc.predict(X_bow_test)
evaluate('LinearSVC', 'BoW', y_test, y_pred)

BoW + LinearSVC: Accuracy = 0.8568


{'model_name': 'LinearSVC',
 'vectorizer': 'BoW',
 'accuracy': 0.85675,
 'y_pred': array([1, 0, 1, ..., 1, 0, 0], dtype=int64)}

### TF-IDF Vectorization Technique

In [71]:
lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(X_tfidf_train, y_train)
y_pred = lr_tfidf.predict(X_tfidf_test)
evaluate('Logistic Regression', 'TF-IDF', y_test, y_pred)

TF-IDF + Logistic Regression: Accuracy = 0.8901


{'model_name': 'Logistic Regression',
 'vectorizer': 'TF-IDF',
 'accuracy': 0.890125,
 'y_pred': array([1, 0, 1, ..., 1, 0, 0], dtype=int64)}

In [72]:
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_tfidf_train, y_train)
y_pred = nb_tfidf.predict(X_tfidf_test)
evaluate('Naive Bayes', 'TF-IDF', y_test, y_pred)

TF-IDF + Naive Bayes: Accuracy = 0.8641


{'model_name': 'Naive Bayes',
 'vectorizer': 'TF-IDF',
 'accuracy': 0.864125,
 'y_pred': array([1, 0, 1, ..., 1, 1, 0], dtype=int64)}

In [73]:
svc_tfidf = LinearSVC(random_state=42, max_iter=10000, dual='auto')
svc_tfidf.fit(X_tfidf_train, y_train)
y_pred = svc_tfidf.predict(X_tfidf_test)
evaluate('LinearSVC', 'TF-IDF', y_test, y_pred)

TF-IDF + LinearSVC: Accuracy = 0.8871


{'model_name': 'LinearSVC',
 'vectorizer': 'TF-IDF',
 'accuracy': 0.887125,
 'y_pred': array([1, 0, 1, ..., 1, 0, 0], dtype=int64)}

In [74]:
# TF-IDF Bigram + Logistic Regression
lr_tfidf_bi = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf_bi.fit(X_tfidf_bi_train, y_train)
y_pred = lr_tfidf_bi.predict(X_tfidf_bi_test)
evaluate('Logistic Regression', 'TF-IDF Bigram', y_test, y_pred)

TF-IDF Bigram + Logistic Regression: Accuracy = 0.8955


{'model_name': 'Logistic Regression',
 'vectorizer': 'TF-IDF Bigram',
 'accuracy': 0.8955,
 'y_pred': array([1, 0, 1, ..., 1, 0, 0], dtype=int64)}

In [75]:
# TF-IDF Bigram + Naive Bayes
nb_tfidf_bi = MultinomialNB()
nb_tfidf_bi.fit(X_tfidf_bi_train, y_train)
y_pred = nb_tfidf_bi.predict(X_tfidf_bi_test)
evaluate('Naive Bayes', 'TF-IDF Bigram', y_test, y_pred)

TF-IDF Bigram + Naive Bayes: Accuracy = 0.8795


{'model_name': 'Naive Bayes',
 'vectorizer': 'TF-IDF Bigram',
 'accuracy': 0.8795,
 'y_pred': array([1, 0, 1, ..., 1, 0, 0], dtype=int64)}

In [76]:
#TF-IDF Bigram + LinearSVC
svc_tfidf_bi = LinearSVC(random_state=42, max_iter=10000, dual='auto')
svc_tfidf_bi.fit(X_tfidf_bi_train, y_train)
y_pred = svc_tfidf_bi.predict(X_tfidf_bi_test)
evaluate('LinearSVC', 'TF-IDF Bigram', y_test, y_pred)

TF-IDF Bigram + LinearSVC: Accuracy = 0.8979


{'model_name': 'LinearSVC',
 'vectorizer': 'TF-IDF Bigram',
 'accuracy': 0.897875,
 'y_pred': array([1, 0, 1, ..., 1, 0, 0], dtype=int64)}

In [77]:
# TF-IDF Stem + Logistic Regression
lr_stem = LogisticRegression(max_iter=1000, random_state=42)
lr_stem.fit(X_tfidf_stem_train, y_train)
y_pred = lr_stem.predict(X_tfidf_stem_test)
evaluate('Logistic Regression', 'TF-IDF Stem', y_test, y_pred)

TF-IDF Stem + Logistic Regression: Accuracy = 0.8911


{'model_name': 'Logistic Regression',
 'vectorizer': 'TF-IDF Stem',
 'accuracy': 0.891125,
 'y_pred': array([1, 0, 1, ..., 1, 0, 0], dtype=int64)}

In [78]:
# TF-IDF Stem + Naive Bayes
nb_stem = MultinomialNB()
nb_stem.fit(X_tfidf_stem_train, y_train)
y_pred = nb_stem.predict(X_tfidf_stem_test)
evaluate('Naive Bayes', 'TF-IDF Stem', y_test, y_pred)

TF-IDF Stem + Naive Bayes: Accuracy = 0.8565


{'model_name': 'Naive Bayes',
 'vectorizer': 'TF-IDF Stem',
 'accuracy': 0.8565,
 'y_pred': array([1, 0, 1, ..., 1, 1, 0], dtype=int64)}

In [79]:
#TF-IDF Stem + LinearSVC
svc_stem = LinearSVC(random_state=42, max_iter=10000, dual='auto')
svc_stem.fit(X_tfidf_stem_train, y_train)
y_pred = svc_stem.predict(X_tfidf_stem_test)
evaluate('LinearSVC', 'TF-IDF Stem', y_test, y_pred)

TF-IDF Stem + LinearSVC: Accuracy = 0.8875


{'model_name': 'LinearSVC',
 'vectorizer': 'TF-IDF Stem',
 'accuracy': 0.8875,
 'y_pred': array([1, 0, 1, ..., 1, 0, 0], dtype=int64)}

In [80]:
results_list = []
for r in results:
    results_list.append({
        'model_name': r['model_name'],
        'vectorizer': r['vectorizer'],
        'accuracy': r['accuracy']
    })

results_df = pd.DataFrame(results_list)
print(results_df)

print("Max Accuracy Model is")
print(results_df.loc[results_df['accuracy'].idxmax()])

             model_name     vectorizer  accuracy
0   Logistic Regression            BoW  0.878750
1           Naive Bayes            BoW  0.856750
2             LinearSVC            BoW  0.856750
3   Logistic Regression         TF-IDF  0.890125
4           Naive Bayes         TF-IDF  0.864125
5             LinearSVC         TF-IDF  0.887125
6   Logistic Regression  TF-IDF Bigram  0.895500
7           Naive Bayes  TF-IDF Bigram  0.879500
8             LinearSVC  TF-IDF Bigram  0.897875
9   Logistic Regression    TF-IDF Stem  0.891125
10          Naive Bayes    TF-IDF Stem  0.856500
11            LinearSVC    TF-IDF Stem  0.887500
Max Accuracy Model is
model_name        LinearSVC
vectorizer    TF-IDF Bigram
accuracy           0.897875
Name: 8, dtype: object


In [81]:
fig = px.bar(
    results_df,
    x='model_name',
    y='accuracy',
    color='vectorizer',
    barmode='group',
    title='Model Comparison — All Vectorizers',
    labels={'accuracy': 'Accuracy', 'model_name': 'Model'},
    text='accuracy'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.add_hline(y=0.85, line_dash='dash', line_color='red', annotation_text='Target: 0.85')
fig.show()

### Key Observations
- TF-IDF consistently outperforms BoW across all models
- Bigrams improve accuracy for all models
- LinearSVC benefits most from bigrams
- Stemming vs Lemmatization: minimal difference overall
- Naive Bayes is fastest but least accurate
- LinearSVC + TF-IDF Bigram is the clear winner

### Best Model
- Model: LinearSVC
- Vectorizer: TF-IDF Bigram (ngram_range=(1,2))
- Accuracy: 0.8979
- Reason: Best accuracy, handles sparse matrices excellently,bigrams capture sentiment phrases like "waste time", "highly recommend"

In [82]:
# Get best model predictions
best_pred = next(r['y_pred'] for r in results  if r['model_name'] == 'LinearSVC' and r['vectorizer'] == 'TF-IDF Bigram')

# Classification Report
print("=== Classification Report ===")
print(classification_report(y_test, best_pred, target_names=['Negative', 'Positive']))

# Confusion Matrix
cm = confusion_matrix(y_test, best_pred)
fig = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Confusion Matrix — LinearSVC + TF-IDF Bigram',
    labels={'x': 'Predicted', 'y': 'Actual'},
    x=['Negative', 'Positive'],
    y=['Negative', 'Positive']
)
fig.show()

=== Classification Report ===
              precision    recall  f1-score   support

    Negative       0.90      0.89      0.90      4000
    Positive       0.89      0.91      0.90      4000

    accuracy                           0.90      8000
   macro avg       0.90      0.90      0.90      8000
weighted avg       0.90      0.90      0.90      8000



## Dense Vectorization + Tree Based Models

### Why Dense Vectorization?

So far we used sparse vectorization (BoW, TF-IDF):
- Each review → vector of 50,000 dimensions
- Mostly zeros (sparse)
- "good" → [0, 0, 0, 1, 0, 0, 0, ...]

Dense vectorization represents each word as a 
compact meaningful vector:
- Each word → vector of 300 dimensions
- No zeros — every dimension carries information
- "good" → [0.23, -0.71, 0.44, ...]
- "great" → [0.21, -0.68, 0.41, ...] (similar to "good"!)


### Why GloVe Pretrained Embeddings?
- Trained on Wikipedia + Gigaword corpus
- Captures semantic meaning out of the box
- No training needed — download and use directly
- "terrible" and "horrible" are close in vector space
- "masterpiece" and "brilliant" are close in vector space
- TF-IDF treats these as completely different words 
- Word2Vec understands they mean similar things 


### How We Use GloVe for Classification?
Each review has multiple words → multiple vectors
We need ONE vector per review for the model:

which means that for we just average the vector of every word in a sentence


### We are using Glove becasue it small in size around 128MB and the Word2Vec is 3.5GB



In [83]:
import gensim.downloader as api

glove_model = api.load('glove-wiki-gigaword-100')

In [84]:
print(f"\nVocabulary size: {len(glove_model)}")
print(f"Vector size: {glove_model.vector_size}")


Vocabulary size: 400000
Vector size: 100


In [85]:
def review_to_vector(review, model, vector_size=100):
    words = review.split()
    # only keep words that exist in glove vocabulary
    vectors = [model[word] for word in words if word in model]
    
    if len(vectors) == 0:
        # if no words found return zeros
        return np.zeros(vector_size)
    
    # average all word vectors
    return np.mean(vectors, axis=0)



In [86]:
print("Converting train reviews to vectors")
X_glove_train = np.array([review_to_vector(review, glove_model) for review in X_lemma_train])

print("Converting test reviews to vectors")
X_glove_test = np.array([review_to_vector(review, glove_model) for review in X_lemma_test])

Converting train reviews to vectors
Converting test reviews to vectors


In [87]:
print(f"\nTrain shape: {X_glove_train.shape}")
print(f"Test shape: {X_glove_test.shape}")


Train shape: (32000, 100)
Test shape: (8000, 100)


In [88]:
svc_glove = LinearSVC(random_state=42, max_iter=10000, dual='auto')
svc_glove.fit(X_glove_train, y_train)
y_pred = svc_glove.predict(X_glove_test)
evaluate('LinearSVC', 'GloVe', y_test, y_pred)

GloVe + LinearSVC: Accuracy = 0.7929


{'model_name': 'LinearSVC',
 'vectorizer': 'GloVe',
 'accuracy': 0.792875,
 'y_pred': array([1, 0, 1, ..., 1, 0, 0], dtype=int64)}

In [89]:
from sklearn.tree import DecisionTreeClassifier

dt_glove = DecisionTreeClassifier(random_state=42)
dt_glove.fit(X_glove_train, y_train)
y_pred = dt_glove.predict(X_glove_test)
evaluate('Decision Tree', 'GloVe', y_test, y_pred)

GloVe + Decision Tree: Accuracy = 0.6631


{'model_name': 'Decision Tree',
 'vectorizer': 'GloVe',
 'accuracy': 0.663125,
 'y_pred': array([0, 0, 1, ..., 1, 1, 1], dtype=int64)}

### Why Dense Vectorization Hurts Accuracy?

#### Problem 1 — Average Pooling Loses Sentiment Signal

Short review: "absolutely terrible waste of time"
- 5 strong negative words averaged
- strong negative signal  - This looks Fine


Long review: "acting was terrible but cinematography was beautiful, soundtrack amazing, story compelling"
- 231 words averaged including positive ones
- negative signal completely diluted


#### Problem 2 — Word2Vec/GloVe Not Trained on Movie Reviews
- GloVe trained on Wikipedia + Gigaword news corpus
- Movie review vocabulary is different:
    - "cinematography", "screenplay", "plot twist"
    - These may have different representations in GloVe

### Conclusion
- Sparse TF-IDF + LinearSVC is clearly better for this task
- Dense embeddings with average pooling lose too much signal
- Decision Tree too simple for 100-dim dense vectors
- Best approach remains: LinearSVC + TF-IDF Bigram → 0.8979

In [ ]:
# lets try to  save the model the vectorizer

import joblib
import os


joblib.dump(tfidf_bigram_vectorizer, '../outputs/models/vectorizer.pkl')
print("Vectorizer saved")

joblib.dump(svc_tfidf_bi, '../outputs/models/model.pkl')
print("Model saved")

loaded_vectorizer = joblib.load('../outputs/models/vectorizer.pkl')
loaded_model = joblib.load('../outputs/models/model.pkl')


sample_reviews = [
    "This movie was absolutely brilliant and amazing!",
    "Terrible waste of time, worst movie ever!"
]


sample_vectorized = loaded_vectorizer.transform(sample_reviews)
sample_preds = loaded_model.predict(sample_vectorized)

for review, pred in zip(sample_reviews, sample_preds):
    sentiment = 'Positive' if pred == 1 else 'Negative'
    print(f"\nReview: {review}")
    print(f"Predicted: {sentiment}")

Vectorizer saved
Model saved


In [91]:
print(sample_vectorized)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 11 stored elements and shape (2, 50000)>
  Coords	Values
  (0, 5329)	0.40345809224458695
  (0, 1676)	0.39678233170011884
  (0, 97)	0.7359427182403278
  (0, 91)	0.3717171835172712
  (1, 48988)	0.2424772370389105
  (1, 47143)	0.28068848609760594
  (1, 44456)	0.5850734139740106
  (1, 44102)	0.13769128403719166
  (1, 43035)	0.6222365685490268
  (1, 43017)	0.2802151131963894
  (1, 13563)	0.18828201440356052


In [92]:
print(sample_preds)

[1 0]
